# B2 · Missing values: native NaN in XGBoost, imputation and indicators

**Type**: Implement with scikit-learn · **Level**: ●●● challenge · **Time**: about 55 minutes · **extended path**

## Objective

Compare in repeated cross-validation three ways of handling missing values with XGBoost on the loans (where the missingness of the income is linked to the outcome): **native NaN**, median imputation, median plus missingness indicators; and with a random forest (which requires imputation). Measure the differences with their standard
error: many are small.

## Dataset

`ds.loans(n=3000)` with missing values in `age`, `income`, `job_tenure` (the missingness of `income` is more frequent among defaulters). Features: `ratio` (amount/income, NaN where income is missing), `age`, `income`, `job_tenure`, `amount`. Validation `RepeatedStratifiedKFold(5, n_repeats=4, random_state=0)`, AUC.

## Educational purpose

Understand how XGBoost "uses" NaN (the default direction of each split) and how much it matters compared with the standard step of imputation + indicator (lesson 2.3, used in exercise B1 of module 4). See that the difference is modest but systematic, and that the estimate has an uncertainty to compute.

## Task

1. Build `models` (dictionary name → `Pipeline` or estimator): `xgb native` (`XGBClassifier` directly, without imputation), `xgb median` (`SimpleImputer(strategy="median")` + XGB), `xgb median+indicators` (`SimpleImputer(strategy="median", add_indicator=True)` + XGB), `rf median+indicators` (same imputer + `RandomForestClassifier(200, min_samples_leaf=5)`). XGB with `n_estimators=150, learning_rate=0.05, max_depth=3, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", n_jobs=-1, random_state=0`.
2. Compute with `cross_val_score(..., scoring="roc_auc", cv=cv)` (20 folds): `table` (index: models; columns `auc` (mean), `sd` (standard deviation across folds)).
3. Paired comparison: for each model other than `xgb native` compute `diff_vs_native` = mean of the fold-by-fold differences `auc(native) − auc(model)`; and `standard_error` (`sd of the differences / √20`). Save `diff` (`pd.DataFrame` with the two columns).
4. Save `native_better_than_median` (True if the mean difference against `xgb median` is positive and exceeds two standard errors) and `indicators_close_gap` (True if `xgb median+indicators` is within one standard error of the native).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline

df = ds.loans(n=3000)
df["ratio"] = df["amount"] / df["income"]                        # NaN where the income is missing
X, y = df[["ratio", "age", "income", "job_tenure", "amount"]], df["defaulted"]
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=0)
print("share of missing values:", X.isna().mean().round(3).to_dict())


def xgb_(): return xgb.XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=3, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", n_jobs=-1, random_state=0)

## Your work

In [ ]:
models = {
    "xgb native": xgb_(),
}

In [ ]:
scores = {}
for name, m in models.items():
    ...
table = pd.DataFrame({n: {"auc": s.mean(), "sd": s.std()} for n, s in scores.items()}).T
table.round(4)

In [ ]:
diff = None
native_better_than_median = None
indicators_close_gap = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`scores[name] = cross_val_score(m, X, y, cv=cv, scoring="roc_auc")`. Paired differences: `d = scores["xgb native"] - scores[name]`; `d.mean()`, `d.std() / np.sqrt(len(d))`. Pipeline: `make_pipeline(SimpleImputer(strategy="median", add_indicator=True), xgb_())`.'):
    check(list(models) == ["xgb native", "xgb median", "xgb median+indicators", "rf median+indicators"] and len(scores) == 4 and all(len(s) == 20 for s in scores.values()), "four models, twenty scores each", "cross_val_score with RepeatedStratifiedKFold(5, 4)")
    check(table["auc"].min() > 0.72 and table["auc"].max() < 0.86, "AUC between 0.72 and 0.86 for all", "the problem has a realistic AUC")
    check(list(diff.index) == ["xgb median", "xgb median+indicators", "rf median+indicators"] and list(diff.columns) == ["diff_vs_native", "standard_error"], "paired differences for the three comparisons", "sd of the differences / sqrt(20)")
    check(diff.loc["xgb median", "diff_vs_native"] > 0 and native_better_than_median is True, "native NaN beat the median alone (by over two standard errors)", "the missingness information is lost with imputation alone")
    check(indicators_close_gap is True and abs(diff.loc["xgb median+indicators", "diff_vs_native"]) < 0.01, "with the missingness indicator the difference from the native disappears (within one standard error, less than one point)", "median + add_indicator is equivalent to the native")
done()

---
The commented solution is in `solutions/B_with_sklearn/B2_native_missing_values_vs_imputation.ipynb`. Open it after you pass the check, or after you have really tried.